# 🗞️ Somali News Scraper
**Categories:** Siyaasad · Amni · Caalamka | **Target:** 10,000 articles per category

| Cell | Purpose |
|------|---------|
| 0    | Install dependencies |
| 1    | Imports & setup |
| 2    | Check current progress |
| 3    | Run full pipeline (all sites × all categories) |
| 4    | Scrape one specific site/category |
| 5    | Inspect raw data |
| 6    | Merge & export final datasets |
| 7    | Quality report |

## Cell 0 — Install Dependencies

In [43]:
# Run once. Restart kernel after installation.
import subprocess, sys

packages = [
    'requests',
    'beautifulsoup4',
    'lxml',
    'pandas',
    'tqdm',
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✔' if result.returncode == 0 else '✘'
    print(f'{status} {pkg}')

print('\n✅ All packages installed. Restart the kernel now if this was your first run.')

✔ requests
✔ beautifulsoup4
✔ lxml
✔ pandas
✔ tqdm

✅ All packages installed. Restart the kernel now if this was your first run.


## Cell 1 — Imports & Setup

In [44]:
import scraper
import pipeline
import utils

print("scraper.py contains:")
print([x for x in dir(scraper) if not x.startswith("_")])

print("\npipeline.py contains:")
print([x for x in dir(pipeline) if not x.startswith("_")])

print("\nutils.py contains:")
print([x for x in dir(utils) if not x.startswith("_")])

scraper.py contains:
['BeautifulSoup', 'CFG', 'SCRAPER_CONFIG', 'USER_AGENTS', 'clean_text', 'extract_article', 'extract_article_links', 'fetch', 'get_logger', 'get_page_url', 'is_empty_listing', 'logger', 'make_session', 'polite_sleep', 'random', 're', 'requests', 'time', 'urljoin', 'urlparse']

pipeline.py contains:
['CATEGORIES', 'CFG', 'Checkpoint', 'DedupCache', 'RawWriter', 'SCRAPER_CONFIG', 'SITES', 'extract_article', 'extract_article_links', 'fetch', 'get_logger', 'get_page_url', 'get_status', 'is_empty_listing', 'logger', 'make_session', 'merge_raw_files', 'polite_sleep', 'print_progress', 'run_pipeline', 'scrape_site_category']

utils.py contains:
['BASE_DIR', 'CHECKPOINT_DIR', 'Checkpoint', 'DATA_DIR', 'DedupCache', 'FINAL_DIR', 'LOG_DIR', 'Path', 'RAW_DIR', 'RawWriter', 'csv', 'd', 'datetime', 'get_logger', 'hashlib', 'json', 'logging', 'merge_raw_files', 'os', 'polite_sleep', 'print_progress', 'random', 'time']


In [52]:
import sys
import os
import warnings
import importlib

warnings.filterwarnings("ignore")

# Make sure project folder is on the path
NOTEBOOK_DIR = os.path.abspath("")
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

import config
import utils
import scraper
import pipeline

# Reload fresh versions, useful after editing .py files
importlib.reload(config)
importlib.reload(utils)
importlib.reload(scraper)
importlib.reload(pipeline)

logger = utils.get_logger()

print("✅ Scraper loaded successfully.")
print(f"   Sites configured : {len(pipeline.SITES)}")
print(f"   Categories       : {pipeline.CATEGORIES}")

✅ Scraper loaded successfully.
   Sites configured : 22
   Categories       : ['siyaasad', 'amni', 'caalamka', 'ciyaaro']


In [46]:
import pipeline
import inspect
import os

print("Current notebook folder:")
print(os.getcwd())

print("\nPipeline file being used:")
print(pipeline.__file__)

print("\nget_status() source currently loaded:")
print(inspect.getsource(pipeline.get_status))

Current notebook folder:
C:\Users\hp\Downloads\files

Pipeline file being used:
C:\Users\hp\Downloads\files\pipeline.py

get_status() source currently loaded:
def get_status() -> dict:
    """Return current article counts per (site, category). Call from notebook."""
    from utils import RAW_DIR
    import csv
    status = {}
    for fp in RAW_DIR.glob("*.csv"):
        stem   = fp.stem                       # e.g. "caasimada__siyaasad"
        parts  = stem.split("__", 1)
        if len(parts) != 2:
            continue
        site, cat = parts
        with open(fp, encoding="utf-8") as f:
            count = max(0, sum(1 for _ in f) - 1)
        status[f"{site}/{cat}"] = count
    return status



In [51]:
import importlib, config, scraper, pipeline
importlib.reload(config); importlib.reload(scraper); importlib.reload(pipeline)

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

site_name = "sonna"
site_cfg  = pipeline.SITES[site_name]
first_url = site_cfg["categories"]["ciyaaro"]
link_sel  = site_cfg["article_link_sel"]

session = scraper.make_session()
r    = session.get(first_url, timeout=15)
soup = BeautifulSoup(r.text, "html.parser")

css_count = len(soup.select(link_sel))
fallback  = sum(1 for a in soup.find_all("a", href=True)
                if scraper._looks_like_article(urljoin(site_cfg["base_url"], a["href"]), site_cfg["base_url"]))

print(f"sonna  {r.status_code}  CSS:{css_count}  fallback:{fallback}")

sonna  200  CSS:7  fallback:64


## Cell 2 — Check Current Progress

In [48]:
import pandas as pd
import pipeline
import utils

status = pipeline.get_status()

if not status:
    print("No data collected yet. Run Cell 3 or Cell 4 to start scraping.")
else:
    rows = []
    for key, count in sorted(status.items()):
        site, cat = key.split("/")
        rows.append({
            "Site": site,
            "Category": cat,
            "Articles": count
        })

    df = pd.DataFrame(rows)

    summary = df.groupby("Category")["Articles"].sum().reset_index()
    summary["Target"] = 10_000
    summary["Progress"] = (summary["Articles"] / summary["Target"] * 100).round(1).astype(str) + "%"
    summary["Remaining"] = (summary["Target"] - summary["Articles"]).clip(lower=0)

    print("\n📊 COLLECTION SUMMARY")
    print("=" * 55)
    print(summary.to_string(index=False))

    print("\n📋 PER-SITE BREAKDOWN")
    print("=" * 55)
    print(
        df.pivot_table(
            index="Site",
            columns="Category",
            values="Articles",
            aggfunc="sum",
            fill_value=0
        ).to_string()
    )

print(f"\n📁 Raw data folder  : {utils.RAW_DIR}")
print(f"📁 Checkpoints      : {utils.CHECKPOINT_DIR}")
print(f"📁 Final datasets   : {utils.FINAL_DIR}")


📊 COLLECTION SUMMARY
Category  Articles  Target Progress  Remaining
    amni      6595   10000    66.0%       3405
caalamka     10116   10000   101.2%          0
 ciyaaro      9024   10000    90.2%        976
siyaasad     10062   10000   100.6%          0

📋 PER-SITE BREAKDOWN
Category        amni  caalamka  ciyaaro  siyaasad
Site                                             
bbc_somali         0         0      245         0
caasimada          0      4049        0      5316
garoweonline       0         0       11         0
goobjoog        6595      1519        0      4746
kooxda             0         0     7193         0
kooxdamanta        0         0        8         0
kubadlive          0         0      114         0
laacibnet          0         0     1094         0
mustaqbalmedia     0      4548       92         0
puntlandpost       0         0      226         0
sonna              0         0       26         0
wararka24          0         0       15         0

📁 Raw data folder  :

## Cell 3 — Run Full Pipeline
> Scrapes **all sites × all categories** automatically. Resumes from checkpoints if interrupted.

In [6]:
import pandas as pd

from pipeline import get_status
from utils import RAW_DIR, CHECKPOINT_DIR, FINAL_DIR

status = get_status()

if not status:
    print("No data collected yet. Run Cell 3 or Cell 4 to start scraping.")
else:
    rows = []
    for key, count in sorted(status.items()):
        site, cat = key.split("/")
        rows.append({
            "Site": site,
            "Category": cat,
            "Articles": count
        })

    df = pd.DataFrame(rows)

    # Summary by category
    summary = df.groupby("Category")["Articles"].sum().reset_index()
    summary["Target"] = 10_000
    summary["Progress"] = (summary["Articles"] / summary["Target"] * 100).round(1).astype(str) + "%"
    summary["Remaining"] = (summary["Target"] - summary["Articles"]).clip(lower=0)

    print("\n📊 COLLECTION SUMMARY")
    print("=" * 55)
    print(summary.to_string(index=False))

    print("\n📋 PER-SITE BREAKDOWN")
    print("=" * 55)
    print(
        df.pivot_table(
            index="Site",
            columns="Category",
            values="Articles",
            aggfunc="sum",
            fill_value=0
        ).to_string()
    )

print(f"\n📁 Raw data folder  : {RAW_DIR}")
print(f"📁 Checkpoints      : {CHECKPOINT_DIR}")
print(f"📁 Final datasets   : {FINAL_DIR}")


📊 COLLECTION SUMMARY
Category  Articles  Target Progress  Remaining
    amni      1548   10000    15.5%       8452
caalamka      5992   10000    59.9%       4008
siyaasad     10062   10000   100.6%          0

📋 PER-SITE BREAKDOWN
Category        amni  caalamka  siyaasad
Site                                    
caasimada          0      4049      5316
goobjoog        1548      1519      4746
mustaqbalmedia     0       424         0

📁 Raw data folder  : C:\Users\hp\Downloads\data\raw
📁 Checkpoints      : C:\Users\hp\Downloads\data\checkpoints
📁 Final datasets   : C:\Users\hp\Downloads\data\final


# Running Full Scraper for Ciyaaro Category only

In [40]:
import importlib, config, scraper, pipeline
importlib.reload(config); importlib.reload(scraper); importlib.reload(pipeline)

CIYAARO_SITES = [
    "kooxda",        # most URLs = most articles, run first
    "kubadlive",
    "laacibnet",
    "goobjoog",
    "wararka24",
    "mustaqbalmedia",
    "garoweonline",
    "sonna",
    "kooxdamanta",
    "puntlandpost",
    "bbc_somali",    # run last — different structure, slower
]

TARGET    = 10_000
MAX_PAGES = 1200

for site in CIYAARO_SITES:
    current = pipeline._count_category_total("ciyaaro")
    if current >= TARGET:
        print(f"✅ Target reached ({current:,}). Done.")
        break

    remaining = TARGET - current
    print(f"\n▶ {site}  |  {remaining:,} still needed")

    n = pipeline.scrape_site_category(
        site_name     = site,
        category      = "ciyaaro",
        target        = remaining,
        max_pages     = MAX_PAGES,
        force_restart = False,
    )
    print(f"  → {n:,} new articles from {site}")

pipeline.merge_raw_files("ciyaaro")
print("\n✅ Ciyaaro collection complete.")

2026-05-19 08:31:10 | INFO     | ▶ [kooxda] [ciyaaro] 8 URL(s) | checkpoint: 0 | writer: 0
2026-05-19 08:31:10 | INFO     |   URL 1/8: https://kooxda.com/category/wararka-ciyaaraha-maanta/



▶ kooxda  |  10,000 still needed


2026-05-19 08:31:12 | INFO     |   [kooxda][ciyaaro] URL 1 Page 0: 20 links
2026-05-19 08:32:18 | INFO     |   [kooxda][ciyaaro] URL 1 Page 1: 68 links
2026-05-19 08:35:44 | INFO     |   [kooxda][ciyaaro] URL 1 Page 2: 16 links
2026-05-19 08:36:37 | INFO     |   [kooxda][ciyaaro] URL 1 Page 3: 16 links
2026-05-19 08:37:31 | INFO     |   [kooxda][ciyaaro] URL 1 Page 4: 18 links
2026-05-19 08:37:34 | INFO     |   ✦ Checkpoint: 100 articles (URL 1, page 4)
2026-05-19 08:38:30 | INFO     |   [kooxda][ciyaaro] URL 1 Page 5: 16 links
2026-05-19 08:39:24 | INFO     |   [kooxda][ciyaaro] URL 1 Page 6: 20 links
2026-05-19 08:40:31 | INFO     |   [kooxda][ciyaaro] URL 1 Page 7: 20 links
2026-05-19 08:41:36 | INFO     |   [kooxda][ciyaaro] URL 1 Page 8: 18 links
2026-05-19 08:42:34 | INFO     |   [kooxda][ciyaaro] URL 1 Page 9: 20 links
2026-05-19 08:43:06 | INFO     |   ✦ Checkpoint: 200 articles (URL 1, page 9)
2026-05-19 08:43:41 | INFO     |   [kooxda][ciyaaro] URL 1 Page 10: 20 links
2026-05

  → 7,193 new articles from kooxda

▶ kubadlive  |  2,807 still needed


2026-05-19 17:42:58 | INFO     |   [kubadlive][ciyaaro] URL 1 Page 0: 9 links
2026-05-19 17:43:14 | WARNING  | Connection error (('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))). Waiting 2.0s… (attempt 1/4)
2026-05-19 17:43:30 | INFO     |   [kubadlive][ciyaaro] URL 1 Page 1: 6 links
2026-05-19 17:43:50 | INFO     |   [kubadlive][ciyaaro] URL 1 Page 2: 9 links
2026-05-19 17:44:29 | INFO     |   [kubadlive][ciyaaro] URL 1 Page 3: 9 links
2026-05-19 17:45:04 | INFO     |   [kubadlive][ciyaaro] URL 1 Page 4: 9 links
2026-05-19 17:45:40 | INFO     |   [kubadlive][ciyaaro] URL 1 Page 5: 9 links
2026-05-19 17:45:43 | WARNING  | Connection error (('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))). Waiting 2.0s… (attempt 1/4)
2026-05-19 17:45:53 | WARNING  | Connection error (('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))). Waiting 2.0s… (attempt 1/4)
2026-05-19

...

  → 226 new articles from puntlandpost

▶ bbc_somali  |  1,221 still needed


2026-05-19 20:30:26 | INFO     |   [bbc_somali][ciyaaro] URL 1 Page 0: 24 links
2026-05-19 20:31:49 | INFO     |   [bbc_somali][ciyaaro] URL 1 Page 1: 16 links
2026-05-19 20:32:44 | INFO     |   [bbc_somali][ciyaaro] URL 1 Page 2: 24 links
2026-05-19 20:34:09 | INFO     |   [bbc_somali][ciyaaro] URL 1 Page 3: 24 links
2026-05-19 20:34:22 | WARNING  | Connection error (('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))). Waiting 2.0s… (attempt 1/4)
2026-05-19 20:35:36 | INFO     |   [bbc_somali][ciyaaro] URL 1 Page 4: 24 links
2026-05-19 20:36:44 | INFO     |   ✦ Checkpoint: 100 articles (URL 1, page 4)
2026-05-19 20:37:01 | INFO     |   [bbc_somali][ciyaaro] URL 1 Page 5: 24 links
2026-05-19 20:38:25 | INFO     |   [bbc_somali][ciyaaro] URL 1 Page 6: 24 links
2026-05-19 20:39:50 | INFO     |   [bbc_somali][ciyaaro] URL 1 Page 7: 24 links
2026-05-19 20:41:15 | INFO     |   [bbc_somali][ciyaaro] URL 1 Page 8: 24 links
2026-05-19 20:42:36 | INFO  

  → 245 new articles from bbc_somali


2026-05-19 20:45:24 | INFO     | [merge] ciyaaro: 9,024 unique articles → ciyaaro_final.csv



✅ Ciyaaro collection complete.


## Cell 4 — Scrape One Specific Site / Category
> Use this to target a specific site or top-up a category that's behind.

In [53]:
import requests
from bs4 import BeautifulSoup

import pipeline
import scraper

SITE = "kubadlive"
CATEGORY = "ciyaaro"

site_cfg = pipeline.SITES[SITE]
cat_url = site_cfg["categories"][CATEGORY]
page_url = scraper.get_page_url(cat_url, 1, site_cfg["pagination"])
link_sel = site_cfg["article_link_sel"]

print("SITE:", SITE)
print("CATEGORY:", CATEGORY)
print("Category URL:", cat_url)
print("Generated Page 1 URL:", page_url)
print("Article link selector:", link_sel)

session = scraper.make_session()

response = session.get(page_url, timeout=20, allow_redirects=True)

print("\nHTTP STATUS:", response.status_code)
print("Final URL:", response.url)
print("Content-Type:", response.headers.get("content-type"))
print("HTML length:", len(response.text))
print("\nFirst 500 characters:")
print(response.text[:500])

soup = BeautifulSoup(response.text, "html.parser")

print("\nPage title:")
print(soup.title.get_text(strip=True) if soup.title else "No title found")

matches = soup.select(link_sel)

print("\nSelector match count:", len(matches))

print("\nFirst matched links:")
for a in matches[:10]:
    print("-", a.get_text(" ", strip=True)[:100], "=>", a.get("href"))

print("\nFallback: all possible article-like links")
candidates = []

for a in soup.select("a[href]"):
    text = a.get_text(" ", strip=True)
    href = a.get("href")

    if text and href and len(text) > 20:
        candidates.append((text, href))

print("Candidate links found:", len(candidates))

for text, href in candidates[:30]:
    print("-", text[:100], "=>", href)

SITE: kubadlive
CATEGORY: ciyaaro
Category URL: https://kubadlive.com/category/transfer/
Generated Page 1 URL: https://kubadlive.com/category/transfer/
Article link selector: h2.entry-title a, h3.entry-title a

HTTP STATUS: 200
Final URL: https://kubadlive.com/category/transfer/
Content-Type: text/html; charset=UTF-8
HTML length: 148558

First 500 characters:
<!doctype html>
<html lang="en-US" prefix="og: https://ogp.me/ns#">
<head>
	
	<meta charset="UTF-8">
	<meta name="viewport" content="width=device-width, initial-scale=1, maximum-scale=5, viewport-fit=cover">
	<link rel="profile" href="https://gmpg.org/xfn/11">

	<link rel="preconnect" href="https://fonts.googleapis.com"><link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<!-- Search Engine Optimization by Rank Math - https://rankmath.com/ -->
<title>Transfer - Wararka Ciyaaraha 

Page title:
Transfer - Wararka Ciyaaraha | Kubada Cagta

Selector match count: 9

First matched links:
- Manchester City Oo Indhaha Ku H

In [8]:
from pathlib import Path
from utils import CHECKPOINT_DIR

dedup = CHECKPOINT_DIR / "dedup__amni.json"
print("Dedup cache exists:", dedup.exists())

Dedup cache exists: True


In [13]:
import importlib, scraper, pipeline
importlib.reload(scraper)
importlib.reload(pipeline)

# Quick sanity check — should print False (page is NOT empty)
import requests
from bs4 import BeautifulSoup
session = scraper.make_session()
r = session.get("https://goobjoog.com/qayb/amniga/", timeout=20)
soup = BeautifulSoup(r.text, "html.parser")
result = scraper.is_empty_listing(soup, ".jeg_post_title a", "https://goobjoog.com")
print("is_empty_listing returned:", result)   # must print False

is_empty_listing returned: False


In [54]:
import importlib
import config, scraper, pipeline

importlib.reload(config)
importlib.reload(scraper)
importlib.reload(pipeline)

# Now scrape
SITE     = "kubadlive"
CATEGORY = "ciyaaro"
TARGET   = 10000
MAX_PAGES = 1300

print(f"Scraping  {SITE} / {CATEGORY}  (target: {TARGET:,} articles)")

n = pipeline.scrape_site_category(
    site_name=SITE,
    category=CATEGORY,
    target=TARGET,
    max_pages=MAX_PAGES,
    force_restart=False,
)
print(f"\n✔ Done. {n:,} new articles written in this run.")

2026-05-20 08:13:08 | INFO     | ▶ [kubadlive] [ciyaaro] 1 URL(s) | checkpoint: 114 | writer: 114
2026-05-20 08:13:08 | INFO     |   Skipping URL 1/1 (already completed)
2026-05-20 08:13:08 | INFO     | ✔ [kubadlive][ciyaaro] Done. New: 0 | Total: 114


Scraping  kubadlive / ciyaaro  (target: 10,000 articles)

✔ Done. 0 new articles written in this run.


## Cell 5 — Inspect Raw Data

In [10]:
import pandas as pd
from utils import RAW_DIR

# Load a single raw file to inspect
INSPECT_SITE     = 'caasimada'
INSPECT_CATEGORY = 'siyaasad'

fp = RAW_DIR / f'{INSPECT_SITE}__{INSPECT_CATEGORY}.csv'

if not fp.exists():
    print(f'File not found: {fp}')
    print('Run Cell 3 or Cell 4 first.')
else:
    df = pd.read_csv(fp)
    print(f'📄 {fp.name}')
    print(f'   Rows    : {len(df):,}')
    print(f'   Columns : {list(df.columns)}')
    print(f'   Date range: {df["scraped_at"].min()} → {df["scraped_at"].max()}\n')
    
    print('── Sample titles (first 10) ──────────────────────────────')
    for i, row in df.head(10).iterrows():
        print(f'  [{i+1:02d}] {row["title"]}')

    print(f'\n── Word count distribution ───────────────────────────────')
    print(df['word_count'].describe().round(1).to_string())

📄 caasimada__siyaasad.csv
   Rows    : 5,316
   Columns : ['id', 'site', 'category', 'url', 'title', 'content', 'scraped_at', 'word_count']
   Date range: 2026-05-16 09:10:46 → 2026-05-17 11:02:24

── Sample titles (first 10) ──────────────────────────────
  [01] DF iyo M/Yurub billaabaya wada-hadallo ku saabsan soo-celinta Soomaalida
  [02] Burcad-badeeda oo faa’iido dhaqaale ka helay dagaalka Iran
  [03] Shacabka Muqdisho oo dhibaato ku qaba helitaanka kaarka aqoonsiga ee NIRA
  [04] Sawirro: Madaxweyne Xasan Sheekh oo xarigga ka jaray…
  [05] Daawo Shariif: “Xasan Sheekh wuxuu taagan yahay meeshii uu ku dhiman lahaa”
  [06] Golaha Mustaqbalka oo Xasan Sheekh u aqoonsaday madaxweyne hore
  [07] Wada-hadalladii Xalane oo fashilmay iyo DF oo ku dhawaaqday in dalku galay…
  [08] Xog: Lacago laaluush ah oo garoonka Muqdisho looga qaado dadka u dhoofa…
  [09] Golaha Mustaqbalka oo war cusub kaso saaray qabsoomida banaanbaxa Muqdisho
  [10] DF oo shaacisay inay duqeysay ciidankii Lafta-gar

## Cell 6 — Merge & Export Final Datasets
> Combines all per-site CSVs into one clean file per category, with cross-site deduplication.

In [11]:
import pandas as pd
from scraper.utils import merge_raw_files, FINAL_DIR

CATEGORIES_TO_MERGE = ['siyaasad', 'amni', 'caalamka']

for cat in CATEGORIES_TO_MERGE:
    out = merge_raw_files(cat)
    if out and out.exists():
        df = pd.read_csv(out)
        print(f'  ✔ {cat:<12} → {len(df):>6,} unique articles saved to {out.name}')

print(f'\n📁 Final files in: {FINAL_DIR}')

ModuleNotFoundError: No module named 'scraper.utils'; 'scraper' is not a package

## Cell 7 — Quality Report

In [ ]:
import pandas as pd
from pathlib import Path
from scraper.utils import FINAL_DIR

print('=' * 65)
print('  DATASET QUALITY REPORT')
print('=' * 65)

all_dfs = []
for fp in sorted(FINAL_DIR.glob('*_final.csv')):
    df = pd.read_csv(fp)
    all_dfs.append(df)
    cat = fp.stem.replace('_final', '')

    missing_titles   = df['title'].isna().sum()
    missing_content  = df['content'].isna().sum()
    short_titles     = (df['title'].str.len() < 20).sum()
    short_content    = (df['word_count'] < 50).sum()
    avg_words        = df['word_count'].mean()
    sites_covered    = df['site'].nunique()
    
    print(f'\n  Category       : {cat.upper()}')
    print(f'  Total articles : {len(df):,}')
    print(f'  Sites          : {sites_covered} ({df["site"].value_counts().to_dict()})')
    print(f'  Avg word count : {avg_words:.0f} words')
    print(f'  Missing titles : {missing_titles}')
    print(f'  Missing content: {missing_content}')
    print(f'  Short titles   : {short_titles} (< 20 chars)')
    print(f'  Short content  : {short_content} (< 50 words)')

if all_dfs:
    combined = pd.concat(all_dfs)
    print(f'\n  TOTAL across all categories: {len(combined):,} articles')
    print(f'  Category balance:')
    print(combined['category'].value_counts().to_string())

print('\n' + '=' * 65)